# Lesson 6: Networks with memory — RNN, LSTM and GRU 🔁
**Learning goals**
1. Understand why reading text needs **memory**, and how an **RNN** passes memory along
2. See the **vanishing gradient** problem (why plain RNNs forget)
3. Learn how **LSTM** and **GRU** use **gates** to remember better
4. Train all three on nursery rhymes and compare them

**⏱️ Timing:** RNN by hand 15' | forgetting problem 10' | gates 10' | train & compare 25'

> ⚙️ GPU recommended: **Runtime → Change runtime type → T4 GPU**. It also runs on CPU, just slower.

## 📖 An analogy: reading a story
In Lesson 5, our rhyme writer only remembered **one** previous word. When you read a story, you remember what happened earlier — that's how you know who "she" is on page 50.

An **RNN (Recurrent Neural Network)** reads one piece at a time and carries a **memory note** (the *hidden state*) from step to step:

`memory_new = tanh( W × current_input  +  U × memory_old  +  bias )`

It's like passing a note down a line of students: each student reads a new word, updates the note, and passes it on.

## Part 1: An RNN by hand 🧮

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(1)

words = ["the", "cat", "sat", "on", "the", "mat"]
vocab = sorted(set(words))
one_hot = {w: np.eye(len(vocab))[i] for i, w in enumerate(vocab)}   # each word → a list of 0s with one 1

hidden_size = 4
W = np.random.randn(hidden_size, len(vocab)) * 0.8   # input → memory
U = np.random.randn(hidden_size, hidden_size) * 0.8  # old memory → new memory
b = np.zeros(hidden_size)

h = np.zeros(hidden_size)          # the memory note starts empty
history = []
for w in words:
    h = np.tanh(W @ one_hot[w] + U @ h + b)
    history.append(h)
    print(f"read '{w:3}' → memory = {np.round(h, 2)}")

In [ ]:
plt.imshow(np.array(history).T, cmap="RdBu", vmin=-1, vmax=1)
plt.xticks(range(len(words)), words); plt.yticks(range(hidden_size), [f"memory {i}" for i in range(hidden_size)])
plt.colorbar(); plt.title("the memory note after each word"); plt.show()

### 🧩 Exercise 1
1. "the" appears twice. Is the memory the same both times? Why not?
2. Change the sentence to `["the", "mat", "sat", "on", "the", "cat"]`. Is the final memory different? 
   (Yes! An RNN cares about **order** — unlike a "bag of words".)

## Part 2: The forgetting problem — vanishing gradients 📉
### 📞 Analogy: the telephone game
Whisper a message down a line of 30 people. By the end, it's garbled. Plain RNNs have the same problem **when learning**: the learning signal from the end of a sentence has to travel back through every step, and it gets multiplied by a small number each time…

In [ ]:
steps = np.arange(0, 51)
for factor in [0.5, 0.9, 1.0, 1.1]:
    plt.plot(steps, factor ** steps, label=f"× {factor} each step")
plt.yscale("log"); plt.xlabel("how many words back"); plt.ylabel("strength of the learning signal")
plt.legend(); plt.title("signals shrink (vanish) or blow up (explode)"); plt.show()
print("0.5 multiplied 50 times =", 0.5 ** 50)

With × 0.5, after just 20 words the signal is less than one millionth. So a plain RNN **can't learn** that a word 30 steps ago matters. (× 1.1 has the opposite problem: **exploding** gradients.)

## Part 3: The fix — gates 🚪
### 📓 Analogy: a diary with an editor
An **LSTM (Long Short-Term Memory)** keeps a separate **long-term memory** (the *cell state*), like a diary. Three **gates** (each a number from 0 to 1) decide what happens to it:
| Gate | Question it asks | Example |
|---|---|---|
| **Forget gate** | What should I erase from the diary? | New paragraph about a different character → forget the old name |
| **Input gate** | What new thing is worth writing down? | A new character's name appears → write it down |
| **Output gate** | What should I share right now? | The word "she" appears → bring out the character's name |

Because the diary is updated by **adding**, not repeated multiplying, information (and the learning signal) can survive for many more steps.

A **GRU (Gated Recurrent Unit)** is a simpler LSTM with just **two** gates (*update* and *reset*) and no separate diary. Fewer weights, often almost as good.

Let's see a forget gate in action:

In [ ]:
def sigmoid(x): return 1 / (1 + np.exp(-x))

diary = 1.0   # long-term memory holds something important, e.g. "the hero's name is Mia"
for name, forget_score in [("forget gate mostly closed (keep)", 4.0), ("forget gate open (erase)", -4.0)]:
    f = sigmoid(forget_score)                     # between 0 and 1
    print(f"{name}: f = {f:.2f}")
    m = diary
    for _ in range(30):
        m = f * m                                 # 30 steps of 'keep this much'
    print(f"   after 30 steps the memory is {m:.3f}\n")

## Part 4: Train an RNN, LSTM and GRU — head to head 🏁
We'll train all three to write nursery rhymes **one character at a time**, using exactly the same data and settings.

In [ ]:
import torch, torch.nn as nn, time
device = "cuda" if torch.cuda.is_available() else "cpu"

text = """Twinkle, twinkle, little star, how I wonder what you are. Up above the world so high, like a diamond in the sky.
Mary had a little lamb, its fleece was white as snow. And everywhere that Mary went, the lamb was sure to go.
Humpty Dumpty sat on a wall. Humpty Dumpty had a great fall. All the king's horses and all the king's men could not put Humpty together again.
Jack and Jill went up the hill to fetch a pail of water. Jack fell down and broke his crown, and Jill came tumbling after.
Hey diddle diddle, the cat and the fiddle, the cow jumped over the moon. The little dog laughed to see such sport, and the dish ran away with the spoon.
Row, row, row your boat, gently down the stream. Merrily, merrily, merrily, merrily, life is but a dream.
Baa, baa, black sheep, have you any wool? Yes sir, yes sir, three bags full. One for the master, one for the dame, and one for the little boy who lives down the lane.
Little Bo Peep has lost her sheep and does not know where to find them. Leave them alone and they will come home, wagging their tails behind them.
Hickory dickory dock, the mouse ran up the clock. The clock struck one, the mouse ran down. Hickory dickory dock.
Rain, rain, go away, come again another day. Little Johnny wants to play.
Old Mother Hubbard went to the cupboard to fetch her poor dog a bone. When she got there the cupboard was bare, and so the poor dog had none.
Little Miss Muffet sat on a tuffet, eating her curds and whey. Along came a spider who sat down beside her and frightened Miss Muffet away.
Little Jack Horner sat in the corner, eating his Christmas pie. He put in his thumb and pulled out a plum and said, what a good boy am I.
The itsy bitsy spider went up the water spout. Down came the rain and washed the spider out. Out came the sun and dried up all the rain, and the itsy bitsy spider went up the spout again.""".lower()
chars = sorted(set(text))
stoi = {c: i for i, c in enumerate(chars)}
data = torch.tensor([stoi[c] for c in text], device=device)
print(f"{len(text)} characters, {len(chars)} different symbols, device = {device}")

In [ ]:
class CharModel(nn.Module):
    def __init__(self, kind, hidden=128):
        super().__init__()
        self.embed = nn.Embedding(len(chars), 32)
        self.rnn = {"RNN": nn.RNN, "LSTM": nn.LSTM, "GRU": nn.GRU}[kind](32, hidden, batch_first=True)
        self.out = nn.Linear(hidden, len(chars))
    def forward(self, x, state=None):
        h, state = self.rnn(self.embed(x), state)
        return self.out(h), state

def train(kind, steps=600, seq_len=60, batch=32):
    torch.manual_seed(0)
    model = CharModel(kind).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=0.005)
    losses, start = [], time.time()
    for step in range(steps):
        idx = torch.randint(0, len(data) - seq_len - 1, (batch,))
        x = torch.stack([data[i:i + seq_len] for i in idx])
        y = torch.stack([data[i + 1:i + seq_len + 1] for i in idx])   # target = the next character
        logits, _ = model(x)
        loss = nn.functional.cross_entropy(logits.reshape(-1, len(chars)), y.reshape(-1))
        opt.zero_grad(); loss.backward(); opt.step()
        losses.append(loss.item())
    n_params = sum(p.numel() for p in model.parameters())
    print(f"{kind:4}: {n_params:6,} weights, final loss {np.mean(losses[-50:]):.2f}, {time.time() - start:.0f}s")
    return model, losses

models, curves = {}, {}
for kind in ["RNN", "LSTM", "GRU"]:
    models[kind], curves[kind] = train(kind)

In [ ]:
for kind, l in curves.items():
    plt.plot(np.convolve(l, np.ones(20) / 20, mode="valid"), label=kind)
plt.xlabel("training step"); plt.ylabel("loss (lower = better)"); plt.legend(); plt.title("who learns fastest?"); plt.show()

In [ ]:
def write(model, start="mary had", length=200, temperature=0.8):
    model.eval()
    x = torch.tensor([[stoi[c] for c in start]], device=device)
    out, state = start, None
    with torch.no_grad():
        for _ in range(length):
            logits, state = model(x, state)
            probs = torch.softmax(logits[0, -1] / temperature, dim=0)
            nxt = torch.multinomial(probs, 1).item()
            out += chars[nxt]
            x = torch.tensor([[nxt]], device=device)   # feed only the new character; memory carries the rest
    return out

for kind, m in models.items():
    print(f"===== {kind} =====\n{write(m)}\n")

### 🧩 Exercise 2: Judge the writers
1. Which model's writing looks most like real rhymes? Do any of them spell real words? Close quotes and full stops correctly?
2. Compare the number of weights. LSTM has the most (4 sets of weights: 3 gates + candidate), GRU has 3 sets, RNN has 1. Was the extra size worth it?
3. Try a different `start` text and `temperature`.

> ⚠️ Our training text is tiny, so all three will partly **copy** the original rhymes. With such short text the differences can be small — real LSTM advantages show up on long texts where memory really matters.

### 🚀 Challenge
- Increase `steps` to 2000 for all three. Which keeps improving the longest?
- Replace `text` with a longer text you're allowed to use (e.g. a public-domain book from Project Gutenberg, like *Alice in Wonderland*). How do the results change?
- Set `seq_len=10`. The model now only practises on 10-character snippets. What happens to its ability to finish words and lines?

## 🔭 Where RNNs fit in history
| Years | Main text model | Memory |
|---|---|---|
| 1990s–2014 | Plain RNN | Short — forgets after a few words |
| 1997 → 2017 | **LSTM / GRU** | Much longer — powered Google Translate, Siri and phone keyboards for years |
| 2017 → today | **Transformer** (Lessons 7–8) | Looks at **all** words at once with attention — no telephone game at all! |

RNNs read **one word at a time, in order**, which is slow to train on huge data. Transformers read everything **in parallel** — that's a big reason they took over.

## 📌 Summary
- **RNN** = a network with a memory note, updated at every step
- Plain RNNs forget because learning signals **vanish** over many steps
- **LSTM** (3 gates + a diary) and **GRU** (2 gates) control what to keep, add and forget
- Next: Transformers throw away step-by-step memory and use **attention** instead